In [1]:
config_path = "config/params.yaml"
mode = "default"

In [2]:
# Parameters
config_path = "config/params.yaml"


# 06 — Fitur per jendela

**Tujuan:** menghitung fitur `docs/05` untuk tiap jendela dan jam, tanpa label.

In [3]:
from concurrent.futures import ProcessPoolExecutor

import pandas as pd
import polars as pl

from ghremoval.config import load_context
from ghremoval.features.behaviour import BEHAVIOUR_COLUMNS, PROXY_COLUMNS, build_hour_features, merge_windows
from ghremoval.quality import Checks
from ghremoval.runlog import RunLog

ctx = load_context(config_path)
run = RunLog("06_features", ctx.paths.run_log, ctx.paths.root)
checks = Checks()
windows = ctx.p.locked["windows_minutes"]
deployable = ctx.p.locked["cohorts"]["deployable_min_events"]
hours = [hour for hour, _ in ctx.all_hours()]

## 1. Hitung per jam, paralel

In [4]:
n = len(hours)
with ProcessPoolExecutor(max_workers=4) as pool:
    stats = list(pool.map(build_hour_features, [ctx.paths.interim] * n, [ctx.paths.processed] * n, hours, [windows] * n, [deployable] * n))
merge_windows(ctx.paths.processed, windows)
table = pd.DataFrame(stats)
table

,hour_id,w5,w10,w15,w30,w60
0,2024-04-01-15,2670,4459,6118,10498,64549
1,2024-04-09-3,1622,2864,3810,6372,42011
2,2024-04-09-15,2664,4669,6360,10957,71169
3,2024-04-11-3,1581,2639,3543,6064,40979
4,2024-04-11-15,2669,4664,6423,11260,71418
5,2024-04-13-3,1257,2062,2732,4624,28298
6,2024-04-13-15,1917,3172,4269,7189,47071
7,2024-04-16-3,1769,2924,3917,6694,43349
8,2024-04-16-15,2938,5062,6867,11866,74830
9,2024-04-18-3,1933,3253,4504,7675,42119


## Ringkasan & cek

In [5]:
per_window = {}
for w in windows:
    frame = pl.scan_parquet(ctx.paths.processed / f"features/window={w}/part.parquet")
    summary = frame.select(pl.len().alias("n"), pl.sum_horizontal(pl.col(BEHAVIOUR_COLUMNS).is_null().sum()).alias("behaviour_null"), pl.sum_horizontal(pl.col(PROXY_COLUMNS).is_null().sum()).alias("proxy_null"),
                           pl.sum_horizontal(pl.col(BEHAVIOUR_COLUMNS + PROXY_COLUMNS).is_nan().sum()).alias("nan")).collect().row(0, named=True)
    per_window[w] = summary
    checks.add(f"Q11_w{w}", summary["behaviour_null"] == 0 and summary["proxy_null"] == 0 and summary["nan"] == 0, str(summary))
checks.raise_if_failed()
print(run.finish(**{f"rows_w{w}": v["n"] for w, v in per_window.items()}))

[LULUS] Q11_w5: {'n': 53460, 'behaviour_null': 0, 'proxy_null': 0, 'nan': 0}
[LULUS] Q11_w10: {'n': 91184, 'behaviour_null': 0, 'proxy_null': 0, 'nan': 0}
[LULUS] Q11_w15: {'n': 123740, 'behaviour_null': 0, 'proxy_null': 0, 'nan': 0}
[LULUS] Q11_w30: {'n': 213756, 'behaviour_null': 0, 'proxy_null': 0, 'nan': 0}


[LULUS] Q11_w60: {'n': 1277605, 'behaviour_null': 0, 'proxy_null': 0, 'nan': 0}
{'notebook': '06_features', 'finished_at_utc': '2026-11-10T05:56:34+00:00', 'git_commit': 'ee7014a', 'duration_seconds': 17.9, 'counts': {'rows_w5': 53460, 'rows_w10': 91184, 'rows_w15': 123740, 'rows_w30': 213756, 'rows_w60': 1277605}}
